In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [9]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content='What is the capital of the moon?', additional_kwargs={}, response_metadata={}, id='799bfcd1-cf59-4965-ac10-2ccbeaa450e0'),
              AIMessage(content='The capital of the moon is Lunapolis.', additional_kwargs={}, response_metadata={}, id='ea17a0df-0e8f-43c6-86ce-3fb133aa4675', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content='What is the weather in Lunapolis?', additional_kwargs={}, response_metadata={}, id='c6cbab42-c508-42dc-83f0-f98f2385fed0'),
              AIMessage(content='Skies are clear, with a high of 120C and a low of -100C.', additional_kwargs={}, response_metadata={}, id='744e6ead-3c20-41d3-8b5c-c864c1a4183b', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content='How many cheese miners live in Lunapolis?', additional_kwargs={}, response_metadata={}, id='f35736c3-9648-4e6d-87eb-d5e786e8a717'),
              AIMessage(content='There are 100,000 cheese miners living in Lunapolis.', addition

In [10]:
print(response["messages"][0].content)

What is the capital of the moon?


## Trim/delete messages

In [2]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [5]:
agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [8]:
from langchain.messages import HumanMessage, AIMessage,ToolMessage
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

print(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='d8aae063-235f-458b-9816-8d2592b5f777'), AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='3fd8284a-9a95-4a7c-b466-de8a3aa82a0f', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='e5d1dbc1-62f7-4089-8aeb-738efd5d9560'), AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='9bb96f11-8f8c-4a9e-b3e9-bc6817ef0926', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='017a8b25-1533-478e-89af-21ec679b95e1'), AIMessage(content='I can’t measure your device’s temperature from here. If it feels hot, that’s a red flag. Here’s what you can do:\n\n- Power down and

In [ ]:
print(response["messages"][-1].content)